##### DatetimeOutParser
%Y 네자리연도, %y 2자리연도, %m 두자리 월 %d 두자리 일

In [1]:
from langchain_classic.output_parsers import DatetimeOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-OutputParser-Date-Enum")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-OutputParser-Date-Enum


In [2]:
output_parser = DatetimeOutputParser()
output_parser.format = "%Y-%m-%d"

print(output_parser.get_format_instructions())

Write a datetime string that matches the following pattern: '%Y-%m-%d'.

Examples: 2026-09-16, 2025-09-16, 2026-09-15

Return ONLY this string, no other words!


In [4]:
template = """Answer the users question:

#Format Instructions:
{format_instructions}

#Question:
{question}

#Answer:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={
        "format_instructions": output_parser.get_format_instructions()
    },
)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={'format_instructions': "Write a datetime string that matches the following pattern: '%Y-%m-%d'.\n\nExamples: 2026-09-16, 2025-09-16, 2026-09-15\n\nReturn ONLY this string, no other words!"}, template='Answer the users question:\n\n#Format Instructions:\n{format_instructions}\n\n#Question:\n{question}\n\n#Answer:')

In [6]:
chain = prompt | ChatOpenAI() | output_parser

output = chain.invoke({"question": "Google이 창업한 연도"})

In [7]:
output.strftime("%Y-%m-%d")

'1998-09-04'

#### 열거형 출력 파서


In [8]:
from enum import Enum
from langchain_classic.output_parsers.enum import EnumOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("CH03-Enum-OutPut-Parser")

LangSmith 추적을 시작합니다.
[프로젝트명]
CH03-Enum-OutPut-Parser


In [9]:
class Colors(Enum):
    RED = "빨간색"
    GREEN = "초록색"
    BLUE = "파란색"

In [10]:
Colors.RED

<Colors.RED: '빨간색'>

In [11]:
parser = EnumOutputParser(enum=Colors)
parser.get_format_instructions()

'Select one of the following options: 빨간색, 초록색, 파란색'

In [12]:
prompt = PromptTemplate.from_template(
    """다음의 물체는 어떤 색깔인가요?
    
    Object: {object}
    
    Instructions: {instructions}"""
).partial(instructions=parser.get_format_instructions())

chain = prompt | ChatOpenAI() | parser

In [13]:
response =chain.invoke({"object": " 하늘"})
print(response)

Colors.BLUE


In [14]:
type(response)

<enum 'Colors'>

In [15]:
response.value

'파란색'